# New York City Yellow Taxi Data

Exploratory data analysis of NYC yellow taxi trips, with a focus on demand patterns, trip characteristics, revenue, and operational opportunities.

## Objective
Use exploratory data analysis (EDA) to understand NYC yellow taxi trip behaviour and identify patterns that can inform operational planning, service efficiency, revenue, and passenger experience.

## Problem Statement
Analyse 2023 NYC yellow taxi trip records to identify when and where demand is concentrated, how trip and fare characteristics vary, and which data-driven actions could improve fleet positioning and operations.

## Analysis Workflow
1. Load and sample trip records
2. Clean data and assess data quality
3. Explore temporal, geographic, and fare patterns
4. Visualise key relationships and demand distributions
5. Summarise findings and operational recommendations

---

---

## Data Understanding
The yellow taxi trip records include fields capturing pick-up and drop-off dates/times, pick-up and drop-off locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts.

The data is stored in Parquet format (*.parquet*). The dataset is from 2009 to 2024. However, this analysis focuses on records from 2023.

The data for each month is present in a different parquet file. The 2023 data is split into twelve monthly files.

The data was collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers like vendors and taxi hailing apps. <br>

You can find the link to the TLC trip records page here: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

The NYC TLC page documents the source data and its schema.

**Trip Records**



|Field Name       |description |
|:----------------|:-----------|
| VendorID | A code indicating the TPEP provider that provided the record. <br> 1= Creative Mobile Technologies, LLC; <br> 2= VeriFone Inc. |
| tpep_pickup_datetime | The date and time when the meter was engaged.  |
| tpep_dropoff_datetime | The date and time when the meter was disengaged.   |
| Passenger_count | The number of passengers in the vehicle. <br> This is a driver-entered value. |
| Trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged |
|RateCodeID |The final rate code in effect at the end of the trip.<br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br>4 = Nassau or Westchester <br>5 = Negotiated fare <br>6 = Group ride |
|Store_and_fwd_flag |This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server.  <br>Y= store and forward trip <br>N= not a store and forward trip |
|Payment_type| A numeric code signifying how the passenger paid for the trip. <br> 1 = Credit card <br>2 = Cash <br>3 = No charge <br>4 = Dispute <br>5 = Unknown <br>6 = Voided trip |
|Fare_amount| The time-and-distance fare calculated by the meter. <br>Extra Miscellaneous extras and surcharges.  Currently, this only includes the 0.50 and 1 USD rush hour and overnight charges. |
|MTA_tax |0.50 USD MTA tax that is automatically triggered based on the metered rate in use. |
|Improvement_surcharge | 0.30 USD improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
|Tip_amount |Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| Tolls_amount | Total amount of all tolls paid in trip.  |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
|Congestion_Surcharge |Total amount collected in trip for NYS congestion surcharge. |
| Airport_fee | 1.25 USD for pick up only at LaGuardia and John F. Kennedy Airports|

Although the amounts of extra charges and taxes applied are specified in the data dictionary, you will see that some cases have different values of these charges in the actual data.

**Taxi Zones**

Each of the trip records contains a field corresponding to the location of the pickup or drop-off of the trip, populated by numbers ranging from 1-263.

These numbers correspond to taxi zones, which may be downloaded as a table or map/shapefile and matched to the trip records using a join.

This is covered in more detail in later sections.

---

## Data Preparation

<br>

### Import Libraries

In [ ]:
#  Import warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%pip install numpy pandas matplotlib seaborn

In [ ]:
%pip install pyarrow

In [ ]:
# Import the libraries you will be using for analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import os
from pathlib import Path

In [ ]:
# Recommended versions
# numpy version: 1.26.4
# pandas version: 2.2.2
# matplotlib version: 3.10.0
# seaborn version: 0.13.2

# Check versions
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)

### Load the dataset
<br>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define constants
BASE_PATH = Path("/content/drive/MyDrive/Colab Notebooks/EDA-NY-Taxi-Analysis")
TRIP_RECORDS = 'trip_records'
YEARLY_SAMPLED_DATA = 'yearly_sampled_data.csv'

You will see twelve files, one for each month.

To read parquet files with Pandas, you have to follow a similar syntax as that for CSV files.

`df = pd.read_parquet('file.parquet')`

In [ ]:
# Try loading one file

df = pd.read_parquet(BASE_PATH/TRIP_RECORDS/"2023-1.parquet")
df.tpep_pickup_datetime
df.info()
df

How many rows are there? Do you think handling such a large number of rows is computationally feasible when we have to combine the data for all twelve months into one?

To handle this, we need to sample a fraction of data from each of the files. How to go about that? Think of a way to select only some portion of the data from each month's file that accurately represents the trends.

#### Sampling the Data
> One way is to take a small percentage of entries for pickup in every hour of a date. So, for all the days in a month, we can iterate through the hours and select 5% values randomly from those. Use `tpep_pickup_datetime` for this. Separate date and hour from the datetime values and then for each date, select some fraction of trips for each of the 24 hours.

To sample data, you can use the `sample()` method. Follow this syntax:

```Python
# sampled_data is an empty DF to keep appending sampled data of each hour
# hour_data is the DF of entries for an hour 'X' on a date 'Y'

sample = hour_data.sample(frac = 0.05, random_state = 42)
# sample 0.05 of the hour_data
# random_state is just a seed for sampling, you can define it yourself

sampled_data = pd.concat([sampled_data, sample]) # adding data for this hour to the DF
```

This *sampled_data* will contain 5% values selected at random from each hour.

Note that the code given above is only the part that will be used for sampling and not the complete code required for sampling and combining the data files.

Keep in mind that you sample by date AND hour, not just hour. (Why?)

---

<br>
Figure out how to sample and combine the files.

**Note:** It is not mandatory to use the method specified above. While sampling, you only need to make sure that your sampled data represents the overall data of all the months accurately.

In [ ]:
# Sample the data
# It is recommmended to not load all the files at once to avoid memory overload

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Select the folder having data files
# Create a list of all the twelve files to read
file_list = os.listdir(BASE_PATH/TRIP_RECORDS)
file_list

In [ ]:
# Take a small percentage of entries from each hour of every date.
# Iterating through the monthly data:
#   read a month file -> day -> hour: append sampled data -> move to next hour -> move to next day after 24 hours -> move to next month file
# Create a single dataframe for the year combining all the monthly data

# Select the folder having data files

# Select the folder having data files

# Create a list of all the twelve files to read
file_list = [
    f for f in os.listdir(BASE_PATH/TRIP_RECORDS)
    if f.startswith("2023-")
    and f.endswith(".parquet")
]

print(file_list)

# initialise an empty dataframe
df = pd.DataFrame()


# iterate through the list of files and sample one by one:
for file_name in file_list:
    try:
        # file path for the current file

        # Reading the current file
        df_cf = pd.read_parquet(BASE_PATH/TRIP_RECORDS/file_name)
        hourly_sample = (
        df_cf.groupby([
            df_cf["tpep_pickup_datetime"].dt.date,
            df_cf["tpep_pickup_datetime"].dt.hour
        ], group_keys=False)
        .apply(lambda x: x.sample(frac=0.05, random_state=42))
)
        # We will store the sampled data for the current date in this df by appending the sampled data from each hour to this
        # After completing iteration through each date, we will append this data to the final dataframe.
        df = pd.concat([df, hourly_sample], ignore_index=True)

        # Loop through dates and then loop through every hour of each date

            # Iterate through each hour of the selected date

                # Sample 5% of the hourly data randomly

                # add data of this hour to the dataframe

        # Concatenate the sampled data of all the dates to a single dataframe

    except Exception as e:
        print(f"Error reading file {file_name}: {e}")

After combining the data files into one DataFrame, convert the new DataFrame to a CSV or parquet file and store it to use directly.

Ideally, you can try keeping the total entries to around 250,000 to 300,000.

In [ ]:
# Load the new data file
def save_csv(df):
  df.to_csv(BASE_PATH/YEARLY_SAMPLED_DATA)

# def load_csv():
#   global df
#   df = pd.read_csv(BASE_PATH/YEARLY_SAMPLED_DATA)


In [ ]:
# Store the df in csv/parquet
df = df.sort_values(
    by="tpep_pickup_datetime"
)
save_csv(df)

In [ ]:
 # Select the folder having data files

# # Create a list of all the twelve files to read
# file_list = [
#     f for f in os.listdir(BASE_PATH/TRIP_RECORDS)
#     if f.startswith("2023-")
#     and f.endswith(".parquet")
# ]

# # initialise an empty dataframe
# df = pd.DataFrame()


# # iterate through the list of files and sample one by one:
# for file_name in file_list:
#     try:
#         # file path for the current file
#         # Reading the current file
#         df_cf = pd.read_parquet(BASE_PATH/TRIP_RECORDS/file_name)
#         print(f"{df_cf.columns}")
#         # January 2023 has airport_fee column and other months have Airport_fee column, we'll normalize it

#     except Exception as e:
#         print(f"Error reading file {file_name}: {e}")

## Data Cleaning
<br>

Now we can load the new data directly.

In [ ]:
# load_csv()

In [ ]:
df.info()

#### Fixing Columns
<br>

Fix/drop any columns as you seem necessary in the below sections

<br>

Fix the index and drop unnecessary columns

In [ ]:
df

In [ ]:
# Fix the index and drop any columns that are not needed
df = df.reset_index(drop=True)
# The column is named Unnamed when created directly and named unnamed when loaded from csv, so handling case.
df = df.drop(
    columns=df.columns[
        df.columns.str.contains(r"^unnamed", case=False, na=False)
    ],
    errors="ignore"
)
df

<br>
There are two airport fee columns. This is possibly an error in naming columns. Let's see whether these can be combined into a single column.

In [ ]:
# Combine the two airport fee columns
df.columns = df.columns.str.lower()
airport_cols = df.loc[:, df.columns == "airport_fee"]
df["airport_fee"] = airport_cols.bfill(axis=1).iloc[:, 0]
df = df.loc[:, ~df.columns.duplicated()]
df

<br>
Fix columns with negative (monetary) values

In [ ]:
monetary_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee"
]

negative_df = df[
    (df[monetary_cols] < 0).any(axis=1)
]

negative_df.shape

In [ ]:
negative_df.head(20)

In [ ]:

negative_df[monetary_cols].hist(
    figsize=(12, 8),
    bins=50,
    log= True
)

plt.tight_layout()
plt.show()

Did you notice something different in the `RatecodeID` column for above records?

In [ ]:
# Analyse RatecodeID for the negative fare amounts
negative_df['ratecodeid'].value_counts().sort_index()


In [ ]:
# Find which columns have negative values
negative_counts = (
    df.select_dtypes(include="number") < 0
).sum()
negative_counts[negative_counts > 0]

In [ ]:
# fix these negative values
df = df[
    (df["fare_amount"] >= 0) &
    (df["total_amount"] >= 0)
].copy()


In [ ]:
df.shape

### Handling Missing Values
<br>

<br>
Find the proportion of missing values in each column




In [ ]:
# missing all data
df.isnull().sum().sum()

In [ ]:
# Find the proportion of missing values in each column
df.isnull().sum()


In [ ]:
datetime_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
]

for col in datetime_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

<br>
Handling missing values in `passenger_count`

In [ ]:
# Display the rows with null values
# Impute NaN values in 'passenger_count'
df[df['passenger_count'].isna()]


In [ ]:
df["passenger_count"].value_counts(dropna=False).sort_index()

In [ ]:
df["passenger_count"].median()
passenger_count_mode = df["passenger_count"].mode()[0]
missing_pct = (
    df["passenger_count"].isna().sum()
    / len(df)
    * 100
)

print(missing_pct)

In [ ]:
df['passenger_count'] = (df['passenger_count'].fillna(passenger_count_mode))

In [ ]:
df["passenger_count"].value_counts(dropna=False).sort_index()

In [ ]:
# Handle zeros
zero_count = (df["passenger_count"] == 0).sum()

print(zero_count)
print(round(zero_count / len(df) * 100, 2))

In [ ]:
df[
    df["passenger_count"] == 0
].sample(20, random_state=42)

In [ ]:
zero_passenger_df = df[
    df["passenger_count"] == 0
]
zero_by_ratecode = (
    zero_passenger_df
    .groupby("ratecodeid")
    .size()
    .sort_values(ascending=False)
)

zero_by_ratecode

In [ ]:
zero_by_vendorcode = (
    zero_passenger_df
    .groupby("vendorid")
    .size()
    .sort_values(ascending=False)
)

zero_by_vendorcode

In [ ]:

df["passenger_count"] = (
    df["passenger_count"]
    .replace(0, df["passenger_count"].mode()[0])
)

In [ ]:
df[df["passenger_count"] == 0][
    ["trip_distance", "fare_amount", "total_amount"]
].describe()

Did you find zeroes in passenger_count? Handle these.

<br>
Handle missing values in `RatecodeID`

In [ ]:
# Fix missing values in 'RatecodeID'
print(df["ratecodeid"].value_counts(dropna=False).sort_index())
df["ratecodeid"].mode()


In [ ]:
ratecode_mode = df["ratecodeid"].mode()[0]

df["ratecodeid"] = (
    df["ratecodeid"]
    .fillna(ratecode_mode)
)

In [ ]:
# save_csv()

<br>
Impute NaN in `congestion_surcharge`

In [ ]:
# handle null values in congestion_surcharge
df["congestion_surcharge"].value_counts(dropna=False)
# df[
#     df["congestion_surcharge"] == 0.5
# ]


In [ ]:
pd.crosstab(
    df["airport_fee"],
    df["congestion_surcharge"]
)

In [ ]:
df.loc[
    df["congestion_surcharge"] == 0.5,
    "congestion_surcharge"
] = 0.0

df["congestion_surcharge"] = (
    df["congestion_surcharge"]
    .fillna(0)
)

Are there missing values in other columns? Did you find NaN values in some other set of columns? Handle those missing values below.

In [ ]:
# Handle any remaining missing values
df.isna().sum()


In [ ]:
# How to handle store-and-flag
df["store_and_fwd_flag"].value_counts(dropna=False)

In [ ]:
store_mode = df["store_and_fwd_flag"].mode()[0]

df["store_and_fwd_flag"] = (
    df["store_and_fwd_flag"]
    .fillna(store_mode)
)

In [ ]:
# Now we handle airport fee
df["airport_fee"].value_counts(dropna=False)

In [ ]:
df["airport_fee"] = (
    df["airport_fee"]
    .fillna(0)
)

In [ ]:
df.isna().sum()

In [ ]:
save_csv(df)

### Handling Outliers
<br>

Before we start fixing outliers, let's perform outlier analysis.

In [ ]:
# Describe the data and check if there are any potential outliers present
# Check for potential out of place values in various columns
numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "airport_fee",
    "congestion_surcharge"
]



In [ ]:
df[numeric_cols].describe().T

In [ ]:
df.nlargest(
    20,
    "trip_distance"
)[
    [
        "trip_distance",
        "fare_amount",
        "total_amount",
        "vendorid"
    ]
]

In [ ]:
df["trip_distance"].quantile(
    [0.90, 0.95, 0.99, 0.999, 0.9999]
)

In [ ]:
df[
    df["trip_distance"] > 100
][
    [
        "trip_distance",
        "fare_amount",
        "total_amount",
        "passenger_count"
    ]
].sort_values(
    by="trip_distance",
    ascending=False
)

In [ ]:
(df["trip_distance"] > 100).sum()

In [ ]:
df[df["trip_distance"] > 100]

<br>
Based on the above analysis, it seems that some of the outliers are present due to errors in registering the trips. Fix the outliers.

Some points you can look for:
- Entries where `trip_distance` is nearly 0 and `fare_amount` is more than 300
- Entries where `trip_distance` and `fare_amount` are 0 but the pickup and dropoff zones are different (both distance and fare should not be zero for different zones)
- Entries where `trip_distance` is more than 250  miles.
- Entries where `payment_type` is 0 (there is no payment_type 0 defined in the data dictionary)

These are just some suggestions. You can handle outliers in any way you wish, using the insights from above outlier analysis.

How will you fix each of these values? Which ones will you drop and which ones will you replace?

First, let us remove 7+ passenger counts as there are very less instances.

In [ ]:
# remove passenger_count > 6
df = df[
    df["passenger_count"] <= 6
].copy()

df.reset_index(drop=True, inplace=True)

In [ ]:
# Continue with outlier handling
near_zero_high_fare = df[
    (df["trip_distance"] < 0.1) &
    (df["fare_amount"] > 300)
]

near_zero_high_fare.shape


In [ ]:
df = df[
    ~(
        (df["trip_distance"] < 0.1) &
        (df["fare_amount"] > 300)
    )
].copy()
df

In [ ]:
# Handle incorrect distance and fare as 0
zero_trip_different_zone = df[
    (df["trip_distance"] == 0) &
    (df["fare_amount"] == 0) &
    (df["pulocationid"] != df["dolocationid"])
]

df = df[
    df["fare_amount"] <= 1000
].copy()

df.reset_index(
    drop=True,
    inplace=True
)

zero_trip_different_zone.shape

In [ ]:
df = df[
    ~(
        (df["trip_distance"] == 0) &
        (df["fare_amount"] == 0) &
        (df["pulocationid"] != df["dolocationid"])
    )
].copy()

In [ ]:
long_trip_df = df[
    df["trip_distance"] > 250
]

long_trip_df.shape

In [ ]:
df = df[
    df["trip_distance"] <= 250
].copy()

In [ ]:
df["payment_type"].value_counts().sort_index()

In [ ]:
df = df[
    df["payment_type"] != 0
].copy()

In [ ]:
# Do any columns need standardising?
# we have standarize date column already


In [ ]:
save_csv(df)

## Exploratory Data Analysis
<br>

In [ ]:
df.columns.tolist()

#### General EDA: Finding Patterns and Trends
<br>

<br>
Categorise the varaibles into Numerical or Categorical.
* `VendorID`:
* `tpep_pickup_datetime`:
* `tpep_dropoff_datetime`:
* `passenger_count`:
* `trip_distance`:
* `RatecodeID`:
* `PULocationID`:
* `DOLocationID`:
* `payment_type`:
* `pickup_hour`:
* `trip_duration`:


The following monetary parameters belong in the same category, is it categorical or numerical?


* `fare_amount`
* `extra`
* `mta_tax`
* `tip_amount`
* `tolls_amount`
* `improvement_surcharge`
* `total_amount`
* `congestion_surcharge`
* `airport_fee`

##### Temporal Analysis

<br>
Analyse the distribution of taxi pickups by hours, days of the week, and months.

In [ ]:
# Find and show the hourly trends in taxi pickups
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour


In [ ]:
# Find and show the daily trends in taxi pickups (days of the week)
hourly_pickups = (
    df.groupby("pickup_hour")
      .size()
      .reset_index(name="trip_count")
)

hourly_pickups
plt.figure(figsize=(10, 5))
plt.plot(
    hourly_pickups["pickup_hour"],
    hourly_pickups["trip_count"],
    marker="o"
)

plt.xticks(range(24))
plt.xlabel("Hour of Day")
plt.ylabel("Number of Pickups")
plt.title("Hourly Taxi Pickup Trends")
plt.grid(True)

plt.show()

In [ ]:
# Show the monthly trends in pickups
df["pickup_month"] = (
    df["tpep_pickup_datetime"]
      .dt.month
)
monthly_pickups = (
    df.groupby("pickup_month")
      .size()
      .reset_index(name="trip_count")
)

plt.figure(figsize=(10, 5))

plt.plot(
    monthly_pickups["pickup_month"],
    monthly_pickups["trip_count"],
    marker="o"
)

plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Number of Pickups")
plt.title("Monthly Taxi Pickup Trends")
plt.grid(True)

plt.show()

##### Financial Analysis

Take a look at the financial parameters like `fare_amount`, `tip_amount`, `total_amount`, and also `trip_distance`. Do these contain zero/negative values?

In [ ]:
# Analyse the above parameters

# We have already cleaned the data

Do you think it is beneficial to create a copy DataFrame leaving out the zero values from these?

<br>
Filter out the zero values from the above columns.

**Note:** The distance might be 0 in cases where pickup and drop is in the same zone. Do you think it is suitable to drop such cases of zero distance?

In [ ]:
df_non_zeros = df[
    (df["trip_distance"] > 0) &
    (df["fare_amount"] > 0) &
    (df["tip_amount"] > 0) &
    (df["total_amount"] > 0)
]
df_non_zeros


<br>
Analyse the monthly revenue (`total_amount`) trend

In [ ]:
# Group data by month and analyse monthly revenue
monthly_revenue = (
    df.groupby("pickup_month")["total_amount"]
      .sum()
      .reset_index()
)

monthly_revenue

plt.figure(figsize=(10,5))

plt.plot(
    monthly_revenue["pickup_month"],
    monthly_revenue["total_amount"],
    marker="o"
)

plt.xlabel("Month")
plt.ylabel("Revenue")
plt.title("Monthly Revenue Trend")

plt.grid(True)

plt.show()

<br>
Show the proportion of each quarter of the year in the revenue

In [ ]:
# Calculate proportion of each quarter
df["quarter"] = (
    df["tpep_pickup_datetime"]
      .dt.quarter
)

quarterly_revenue = (
    df.groupby("quarter")["total_amount"]
      .sum()
)

quarterly_pct = (
    quarterly_revenue
    / quarterly_revenue.sum()
    * 100
)

quarterly_pct.plot(
    kind="pie",
    autopct="%1.1f%%"
)


<br>
Visualise the relationship between `trip_distance` and `fare_amount`. Also find the correlation value for these two.

**Hint:** You can leave out the trips with trip_distance = 0

In [ ]:
# Show how trip fare is affected by distance
distance_fare_df = df[
    df["trip_distance"] > 0
]

distance_fare_df[
    ["trip_distance", "fare_amount"]
].corr()

plt.figure(figsize=(8,6))

plt.scatter(
    distance_fare_df["trip_distance"],
    distance_fare_df["fare_amount"],
    alpha=0.3
)

plt.xlabel("Trip Distance")
plt.ylabel("Fare Amount")
plt.title("Trip Distance vs Fare Amount")

plt.show()

In [ ]:
df[
    df["fare_amount"] > 10000
]

<br>
Find and visualise the correlation between:
1. `fare_amount` and trip duration (pickup time to dropoff time)
2. `fare_amount` and `passenger_count`
3. `tip_amount` and `trip_distance`

In [ ]:
# Show relationship between fare and trip duration
df["trip_duration"] = (
    df["tpep_dropoff_datetime"] -
    df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["trip_duration"].describe()


In [ ]:
negative_duration_df = df[
    df["trip_duration"] < 0
]

negative_duration_df.shape

In [ ]:
negative_duration_df[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration",
        "trip_distance",
        "fare_amount"
    ]
].head(20)

In [ ]:
(df["trip_duration"] < 0).sum()

In [ ]:
# Trip duraition more thatn 5 hrs
df[
    df["trip_duration"] > 300
][
    [
        "trip_duration",
        "trip_distance",
        "fare_amount"
    ]
].sort_values(
    "trip_duration",
    ascending=False
).head(20)

In [ ]:
df = df[
    (df["trip_duration"] > 0) &
    (df["trip_duration"] <= 300)
].copy()

df.reset_index(drop=True, inplace=True)

In [ ]:
save_csv(df)

In [ ]:
corr = df["fare_amount"].corr(
    df["trip_duration"]
)

print(corr)

In [ ]:
# Show relationship between fare and number of passengers

fare_by_passenger = (
    df.groupby("passenger_count")["fare_amount"]
      .mean()
      .reset_index()
)

plt.figure(figsize=(8, 5))

plt.bar(
    fare_by_passenger["passenger_count"],
    fare_by_passenger["fare_amount"]
)

plt.xlabel("Passenger Count")
plt.ylabel("Average Fare Amount")
plt.title("Average Fare by Passenger Count")

plt.show()

In [ ]:
# Show relationship between tip and trip distance
tip_distance_corr = df["tip_amount"].corr(
    df["trip_distance"]
)

payment_pct = (
    df["payment_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .sort_index()
)

payment_pct

payment_counts.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.xlabel("Payment Type")
plt.ylabel("Number of Trips")
plt.title("Distribution of Payment Types")

plt.show()


<br>
Analyse the distribution of different payment types (`payment_type`)

In [ ]:
# Analyse the distribution of different payment types (payment_type).
payment_counts = (
    df["payment_type"]
    .value_counts()
    .sort_index()
)

payment_counts




- 1= Credit card
- 2= Cash
- 3= No charge
- 4= Dispute



##### Geographical Analysis

For this, you have to use the *taxi_zones.shp* file from the *taxi_zones* folder.

There would be multiple files inside the folder (such as *.shx, .sbx, .sbn* etc). You do not need to import/read any of the files other than the shapefile, *taxi_zones.shp*.

Do not change any folder structure - all the files need to be present inside the folder for it to work.

The folder structure should look like this:
```
Taxi Zones
|- taxi_zones.shp.xml
|- taxi_zones.prj
|- taxi_zones.sbn
|- taxi_zones.shp
|- taxi_zones.dbf
|- taxi_zones.shx
|- taxi_zones.sbx

 ```

 You only need to read the `taxi_zones.shp` file. The *shp* file will utilise the other files by itself.

We will use the *GeoPandas* library for geopgraphical analysis
```
import geopandas as gpd
```

More about geopandas and shapefiles: [About](https://geopandas.org/en/stable/about.html)


Reading the shapefile is very similar to *Pandas*. Use `gpd.read_file()` function to load the data (*taxi_zones.shp*) as a GeoDataFrame. Documentation: [Reading and Writing Files](https://geopandas.org/en/stable/docs/user_guide/io.html)

In [ ]:
!pip install geopandas

<br>
Load the shapefile and display it.

In [ ]:
import geopandas as gpd

# Read the shapefile using geopandas
zones = gpd.read_file(
    BASE_PATH / "taxi_zones" / "taxi_zones.shp"
)
zones.head()

Now, if you look at the DataFrame created, you will see columns like: `OBJECTID`,`Shape_Leng`, `Shape_Area`, `zone`, `LocationID`, `borough`, `geometry`.
<br>
Now, the `locationID` here is also what we are using to mark pickup and drop zones in the trip records.

The geometric parameters like shape length, shape area and geometry are used to plot the zones on a map.

This can be easily done using the `plot()` method.

In [ ]:
print(zones.info())
zones.plot()

Now, you have to merge the trip records and zones data using the location IDs.



<br>
Merge the zones data into trip data using the `locationID` and `PULocationID` columns.

In [ ]:
# Merge zones and trip records using locationID and PULocationID
pickup_df = df.merge(
    zones,
    left_on="pulocationid",
    right_on="LocationID",
    how="left"
)


<br>
Group data by location IDs to find the total number of trips per location ID

In [ ]:
# Group data by location and calculate the number of trips
pickup_df = df.merge(
    zones,
    left_on="pulocationid",
    right_on="LocationID",
    how="left"
)
trip_counts = (
    pickup_df.groupby(
        ["LocationID", "zone", "borough"]
    )
    .size()
    .reset_index(name="trip_count")
    .sort_values(
        "trip_count",
        ascending=False
    )
)

trip_counts.head(10)

<br>
Now, use the grouped data to add number of trips to the GeoDataFrame.

We will use this to plot a map of zones showing total trips per zone.

In [ ]:
# Merge trip counts back to the zones GeoDataFrame
trip_counts = (
    pickup_df.groupby("LocationID")
    .size()
    .reset_index(name="trip_count")
)

zones_trips = zones.merge(
    trip_counts,
    on="LocationID",
    how="left"
)

zones_trips["trip_count"] = (
    zones_trips["trip_count"]
    .fillna(0)
)


The next step is creating a color map (choropleth map) showing zones by the number of trips taken.

Again, you can use the `zones.plot()` method for this. [Plot Method GPD](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html#geopandas.GeoDataFrame.plot)

But first, you need to define the figure and axis for the plot.

`fig, ax = plt.subplots(1, 1, figsize = (12, 10))`

This function creates a figure (fig) and a single subplot (ax)

---

After setting up the figure and axis, we can proceed to plot the GeoDataFrame on this axis. This is done in the next step where we use the plot method of the GeoDataFrame.

You can define the following parameters in the `zones.plot()` method:
```
column = '',
ax = ax,
legend = True,
legend_kwds = {'label': "label", 'orientation': "<horizontal/vertical>"}
```

To display the plot, use `plt.show()`.

<br>
Plot a color-coded map showing zone-wise trips

In [ ]:
# Define figure and axis


# Plot the map and display it

fig, ax = plt.subplots(
    1, 1,
    figsize=(12, 10)
)

zones_trips.plot(
    column="trip_count",
    ax=ax,
    legend=True,
    legend_kwds={
        "label": "Number of Trips",
        "orientation": "vertical"
    }
)

ax.set_title("Zone-wise Taxi Trips")
ax.axis("off")

plt.show()

In [ ]:
# can you try displaying the zones DF sorted by the number of trips?
zones_trips.sort_values(
    by="trip_count",
    ascending=False
)[
    [
        "LocationID",
        "zone",
        "borough",
        "trip_count"
    ]
]


Here we have completed the temporal, financial and geographical analysis on the trip records.

**Compile your findings from general analysis below:**

You can consider the following points:

* Busiest hours, days and months
* Trends in revenue collected
* Trends in quarterly revenue
* How fare depends on trip distance, trip duration and passenger counts
* How tip amount depends on trip distance
* Busiest zones


#### Detailed EDA: Insights and Strategies
<br>

Having performed basic analyses for finding trends and patterns, we will now move on to some detailed analysis focussed on operational efficiency, pricing strategies, and customer experience.

##### Operational Efficiency

Analyze variations by time of day and location to identify bottlenecks or inefficiencies in routes

<br>
Identify slow routes by calculating the average time taken by cabs to get from one zone to another at different hours of the day.

Speed on a route *X* for hour *Y* = (*distance of the route X / average trip duration for hour Y*)

In [ ]:
# Find routes which have the slowest speeds at different times of the day

route_speed_df = (
    df.groupby(
        [
            "pulocationid",
            "dolocationid",
            "pickup_hour"
        ]
    )
    .agg(
        avg_distance=("trip_distance", "mean"),
        avg_duration=("trip_duration", "mean")
    )
    .reset_index()
)

route_speed_df["avg_speed"] = (
    route_speed_df["avg_distance"] /
    (route_speed_df["avg_duration"] / 60)
)

route_speed_df = (
    df.groupby(
        [
            "pulocationid",
            "dolocationid",
            "pickup_hour"
        ]
    )
    .agg(
        trip_count=("trip_distance", "size"),
        avg_distance=("trip_distance", "mean"),
        avg_duration=("trip_duration", "mean")
    )
    .reset_index()
)

route_speed_df["avg_speed"] = (
    route_speed_df["avg_distance"] /
    (route_speed_df["avg_duration"] / 60)
)


route_speed_df = route_speed_df[
    route_speed_df["trip_count"] >= 20
]

route_speed_df = route_speed_df[
    route_speed_df["pulocationid"] !=
    route_speed_df["dolocationid"]
]

slow_routes = (
    route_speed_df.sort_values(
        by="avg_speed"
    )
)

slow_routes.head(20)

zone_lookup = zones[
    ["LocationID", "zone"]
]

slow_routes = slow_routes.merge(
    zone_lookup,
    left_on="pulocationid",
    right_on="LocationID",
    how="left"
)

slow_routes.rename(
    columns={"zone": "pickup_zone"},
    inplace=True
)

slow_routes = slow_routes.merge(
    zone_lookup,
    left_on="dolocationid",
    right_on="LocationID",
    how="left"
)

slow_routes.rename(
    columns={"zone": "dropoff_zone"},
    inplace=True
)

slow_routes[
    [
        "pickup_zone",
        "dropoff_zone",
        "pickup_hour",
        "trip_count",
        "avg_speed"
    ]
].sort_values(
    "avg_speed"
).head(20)

How does identifying high-traffic, high-demand routes help us?

<br>
Calculate the number of trips at each hour of the day and visualise them. Find the busiest hour and show the number of trips for that hour.

In [ ]:
# Visualise the number of trips per hour and find the busiest hour
df["pickup_hour"] = (
    df["tpep_pickup_datetime"]
    .dt.hour
)
hourly_trips = (
    df.groupby("pickup_hour")
      .size()
      .reset_index(name="trip_count")
      .sort_values("pickup_hour")
)

hourly_trips

plt.figure(figsize=(10, 5))

plt.plot(
    hourly_trips["pickup_hour"],
    hourly_trips["trip_count"],
    marker="o"
)

plt.xlabel("Hour of Day")
plt.ylabel("Number of Trips")
plt.title("Trips by Hour")
plt.xticks(range(24))
plt.grid(True)

plt.show()

In [ ]:
busiest_hour = hourly_trips.loc[
    hourly_trips["trip_count"].idxmax()
]

busiest_hour

Remember, we took a fraction of trips. To find the actual number, you have to scale the number up by the sampling ratio.

<br>
Find the actual number of trips in the five busiest hours

In [ ]:
# Scale up the number of trips

# Fill in the value of your sampling fraction and use that to scale up the numbers
sample_fraction = 0.05

top_5_hours = (
    hourly_trips
    .sort_values("trip_count", ascending=False)
    .head(5)
    .copy()
)

top_5_hours["actual_trip_count"] = (
    top_5_hours["trip_count"] * 20
)

top_5_hours

<br>
Compare hourly traffic pattern on weekdays. Also compare for weekend.

In [ ]:
# Compare traffic trends for the week days and weekends
df["day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6])

weekday_hourly = (
    df[~df["is_weekend"]]
    .groupby("pickup_hour")
    .size()
)

weekend_hourly = (
    df[df["is_weekend"]]
    .groupby("pickup_hour")
    .size()
)

plt.figure(figsize=(12, 6))

plt.plot(
    weekday_hourly.index,
    weekday_hourly.values,
    marker="o",
    label="Weekdays"
)

plt.plot(
    weekend_hourly.index,
    weekend_hourly.values,
    marker="o",
    label="Weekends"
)

plt.xlabel("Hour")
plt.ylabel("Trips")
plt.title("Weekday vs Weekend Hourly Traffic")
plt.xticks(range(24))
plt.legend()
plt.grid(True)

plt.show()

What can you infer from the above patterns? How will finding busy and quiet hours for each day help us?

<br>
Identify top 10 zones with high hourly pickups. Do the same for hourly dropoffs. Show pickup and dropoff trends in these zones.

In [ ]:
# Find top 10 pickup and dropoff zones
top_pickup_zones = (
    df.groupby("pulocationid")
      .size()
      .reset_index(name="pickup_count")
      .sort_values(
          "pickup_count",
          ascending=False
      )
      .head(10)
)

top_pickup_zones = top_pickup_zones.merge(
    zones[["LocationID", "zone"]],
    left_on="pulocationid",
    right_on="LocationID",
    how="left"
)

top_pickup_zones[["zone", "pickup_count"]]




In [ ]:
top_dropoff_zones = (
    df.groupby("dolocationid")
      .size()
      .reset_index(name="dropoff_count")
      .sort_values(
          "dropoff_count",
          ascending=False
      )
      .head(10)
)

top_dropoff_zones = top_dropoff_zones.merge(
    zones[["LocationID", "zone"]],
    left_on="dolocationid",
    right_on="LocationID",
    how="left"
)

top_dropoff_zones[["zone", "dropoff_count"]]

In [ ]:
top_pickup_ids = top_pickup_zones["pulocationid"]

pickup_hourly = (
    df[df["pulocationid"].isin(top_pickup_ids)]
    .groupby(
        ["pulocationid", "pickup_hour"]
    )
    .size()
    .reset_index(name="trip_count")
)

pickup_hourly = pickup_hourly.merge(
    zones[["LocationID", "zone"]],
    left_on="pulocationid",
    right_on="LocationID",
    how="left"
)

In [ ]:

plt.figure(figsize=(14, 8))

for zone in pickup_hourly["zone"].unique():
    temp = pickup_hourly[
        pickup_hourly["zone"] == zone
    ]

    plt.plot(
        temp["pickup_hour"],
        temp["trip_count"],
        label=zone
    )

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.xlabel("Hour")
plt.ylabel("Trips")
plt.title("Hourly Pickup Trends for Top 10 Pickup Zones")

plt.show()

In [ ]:
top_dropoff_ids = top_dropoff_zones["dolocationid"]

dropoff_hourly = (
    df[df["dolocationid"].isin(top_dropoff_ids)]
    .groupby(
        ["dolocationid", "pickup_hour"]
    )
    .size()
    .reset_index(name="trip_count")
)

dropoff_hourly = dropoff_hourly.merge(
    zones[["LocationID", "zone"]],
    left_on="dolocationid",
    right_on="LocationID",
    how="left"
)

plt.figure(figsize=(14, 8))

for zone in dropoff_hourly["zone"].unique():
    temp = dropoff_hourly[
        dropoff_hourly["zone"] == zone
    ]

    plt.plot(
        temp["pickup_hour"],
        temp["trip_count"],
        label=zone
    )

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.xlabel("Hour")
plt.ylabel("Trips")
plt.title("Hourly Dropoff Trends for Top 10 Dropoff Zones")

plt.show()

<br>
Find the ratio of pickups and dropoffs in each zone. Display the 10 highest (pickup/drop) and 10 lowest (pickup/drop) ratios.

In [ ]:
# Find the top 10 and bottom 10 pickup/dropoff ratios

pickup_counts = (
    df.groupby("pulocationid")
      .size()
      .reset_index(name="pickup_count")
)

dropoff_counts = (
    df.groupby("dolocationid")
      .size()
      .reset_index(name="dropoff_count")
)

zone_flow = pickup_counts.merge(
    dropoff_counts,
    left_on="pulocationid",
    right_on="dolocationid",
    how="outer"
)

zone_flow["LocationID"] = (
    zone_flow["pulocationid"]
    .fillna(zone_flow["dolocationid"])
)

zone_flow["pickup_count"] = (
    zone_flow["pickup_count"]
    .fillna(0)
)

zone_flow["dropoff_count"] = (
    zone_flow["dropoff_count"]
    .fillna(0)
)

zone_flow["pickup_dropoff_ratio"] = (
    zone_flow["pickup_count"]
    / zone_flow["dropoff_count"]
)

zone_flow = zone_flow.merge(
    zones[["LocationID", "zone"]],
    on="LocationID",
    how="left"
)

top_10_ratio = (
    zone_flow[
        zone_flow["dropoff_count"] > 0
    ]
    .sort_values(
        "pickup_dropoff_ratio",
        ascending=False
    )
    .head(10)
)

top_10_ratio[
    ["zone",
     "pickup_count",
     "dropoff_count",
     "pickup_dropoff_ratio"]
]

In [ ]:
bottom_10_ratio = (
    zone_flow[
        zone_flow["dropoff_count"] > 0
    ]
    .sort_values(
        "pickup_dropoff_ratio"
    )
    .head(10)
)

bottom_10_ratio[
    ["zone",
     "pickup_count",
     "dropoff_count",
     "pickup_dropoff_ratio"]
]

<br>
Identify zones with high pickup and dropoff traffic during night hours (11PM to 5AM)

In [ ]:
# During night hours (11pm to 5am) find the top 10 pickup and dropoff zones
# Note that the top zones should be of night hours and not the overall top zones

night_df = df[
    df["pickup_hour"].isin([23, 0, 1, 2, 3, 4, 5])
]

night_pickups = (
    night_df.groupby("pulocationid")
            .size()
            .reset_index(name="pickup_count")
            .sort_values(
                "pickup_count",
                ascending=False
            )
            .head(10)
)

night_pickups = night_pickups.merge(
    zones[["LocationID", "zone"]],
    left_on="pulocationid",
    right_on="LocationID",
    how="left"
)

night_pickups[
    ["zone", "pickup_count"]
]


In [ ]:

night_dropoffs = (
    night_df.groupby("dolocationid")
            .size()
            .reset_index(name="dropoff_count")
            .sort_values(
                "dropoff_count",
                ascending=False
            )
            .head(10)
)

night_dropoffs = night_dropoffs.merge(
    zones[["LocationID", "zone"]],
    left_on="dolocationid",
    right_on="LocationID",
    how="left"
)

night_dropoffs[
    ["zone", "dropoff_count"]
]

Now, let us find the revenue share for the night time hours and the day time hours. After this, we will move to deciding a pricing strategy.

<br>
Find the revenue share for nighttime and daytime hours.

In [ ]:
# Filter for night hours (11 PM to 5 AM)
night_hours = [23, 0, 1, 2, 3, 4, 5]
night_df = df[
    df["pickup_hour"].isin(night_hours)
]

day_df = df[
    ~df["pickup_hour"].isin(night_hours)
]
night_revenue = night_df["total_amount"].sum()
day_revenue = day_df["total_amount"].sum()

print(f"Night Revenue: ${night_revenue:,.2f}")
print(f"Day Revenue: ${day_revenue:,.2f}")

In [ ]:
total_revenue = df["total_amount"].sum()

night_pct = (
    night_revenue / total_revenue
) * 100

day_pct = (
    day_revenue / total_revenue
) * 100

print(f"Night Revenue %: {night_pct:.2f}%")
print(f"Day Revenue %: {day_pct:.2f}%")

In [ ]:
revenue_summary = pd.DataFrame({
    "Period": ["Day", "Night"],
    "Revenue": [
        day_revenue,
        night_revenue
    ]
})

revenue_summary["Revenue %"] = (
    revenue_summary["Revenue"]
    / revenue_summary["Revenue"].sum()
    * 100
)

revenue_summary

##### Pricing Strategy

<br>
For the different passenger counts, find the average fare per mile per passenger.

For instance, suppose the average fare per mile for trips with 3 passengers is 3 USD/mile, then the fare per mile per passenger will be 1 USD/mile.

In [ ]:
# Analyse the fare per mile per passenger for different passenger counts

fare_df = df[
    df["trip_distance"] > 0
].copy()

fare_df["fare_per_mile_per_passenger"] = (
    fare_df["fare_amount"]
    / (
        fare_df["trip_distance"]
        * fare_df["passenger_count"]
    )
)

fare_per_passenger = (
    fare_df.groupby("passenger_count")[
        "fare_per_mile_per_passenger"
    ]
    .mean()
    .reset_index()
)

fare_per_passenger

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    fare_per_passenger["passenger_count"],
    fare_per_passenger["fare_per_mile_per_passenger"]
)

plt.xlabel("Passenger Count")
plt.ylabel("Average Fare Per Mile Per Passenger ($)")
plt.title("Fare Per Mile Per Passenger")

plt.show()

<br>
Find the average fare per mile by hours of the day and by days of the week

In [ ]:
# Compare the average fare per mile for different days and for different times of the day
fare_df = df[
    df["trip_distance"] > 0
].copy()

fare_df["fare_per_mile"] = (
    fare_df["fare_amount"]
    / fare_df["trip_distance"]
)
fare_df["day_name"] = (
    fare_df["tpep_pickup_datetime"]
    .dt.day_name()
)
hourly_fare = (
    fare_df.groupby("pickup_hour")
           ["fare_per_mile"]
           .mean()
           .reset_index()
)

hourly_fare

In [ ]:
hourly_fare.plot(
    x="pickup_hour",
    y="fare_per_mile",
    kind="line",
    marker="o",
    figsize=(10,5)
)

In [ ]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

daily_fare = (
    fare_df.groupby("day_name")
           ["fare_per_mile"]
           .mean()
           .reindex(day_order)
           .reset_index()
)

daily_fare



In [ ]:
daily_fare.plot(
    x="day_name",
    y="fare_per_mile",
    kind="bar",
    figsize=(10,5)
)

<br>
Analyse the average fare per mile for the different vendors for different hours of the day

In [ ]:
# Compare fare per mile for different vendors

vendor_hourly = (
    fare_df.groupby(
        ["vendorid", "pickup_hour"]
    )["fare_per_mile"]
     .mean()
     .reset_index()
)

plt.figure(figsize=(12,6))

for vendor in vendor_hourly["vendorid"].unique():
    temp = vendor_hourly[
        vendor_hourly["vendorid"] == vendor
    ]

    plt.plot(
        temp["pickup_hour"],
        temp["fare_per_mile"],
        label=f"Vendor {vendor}"
    )

plt.legend()
plt.xlabel("Hour")
plt.ylabel("Fare Per Mile")
plt.title("Fare Per Mile by Vendor and Hour")
plt.grid(True)

plt.show()

<br>
Compare the fare rates of the different vendors in a tiered fashion. Analyse the average fare per mile for distances upto 2 miles. Analyse the fare per mile for distances from 2 to 5 miles. And then for distances more than 5 miles.


In [ ]:
df.columns

In [ ]:
# Defining distance tiers
fare_df["distance_tier"] = pd.cut(
    fare_df["trip_distance"],
    bins=[0, 2, 5, float("inf")],
    labels=["0-2 Miles", "2-5 Miles", "5+ Miles"]
)

fare_df["fare_per_mile"] = (
    fare_df["fare_amount"]
    / fare_df["trip_distance"]
)


tier_vendor = (
    fare_df.groupby(
        ["distance_tier", "vendorid"]
    )["fare_per_mile"]
    .mean()
    .reset_index()
)

tier_vendor



In [ ]:
tier_vendor_pivot = tier_vendor.pivot(
    index="distance_tier",
    columns="vendorid",
    values="fare_per_mile"
)

tier_vendor_pivot

In [ ]:
tier_vendor_pivot.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Average Fare Per Mile by Vendor and Distance Tier")
plt.ylabel("Fare Per Mile ($)")
plt.xlabel("Distance Tier")
plt.grid(axis="y")

plt.show()

##### Customer Experience and Other Factors

<br>
Analyse average tip percentages based on trip distances, passenger counts and time of pickup. What factors lead to low tip percentages?

In [ ]:
#  Analyze tip percentages based on distances, passenger counts and pickup times
tip_df = df[
    df["fare_amount"] > 0
].copy()

tip_df["tip_percentage"] = (
    tip_df["tip_amount"]
    / tip_df["fare_amount"]
) * 100
tip_df["distance_tier"] = pd.cut(
    tip_df["trip_distance"],
    bins=[0, 2, 5, float("inf")],
    labels=["0-2 Miles", "2-5 Miles", "5+ Miles"]
)

distance_tips = (
    tip_df.groupby("distance_tier")
          ["tip_percentage"]
          .mean()
          .reset_index()
)

distance_tips

In [ ]:
distance_tips.plot(
    x="distance_tier",
    y="tip_percentage",
    kind="bar",
    figsize=(8,5)
)

plt.ylabel("Average Tip %")
plt.title("Average Tip Percentage by Distance")
plt.show()

In [ ]:
passenger_tips = (
    tip_df.groupby("passenger_count")
          ["tip_percentage"]
          .mean()
          .reset_index()
)

passenger_tips

passenger_tips.plot(
    x="passenger_count",
    y="tip_percentage",
    kind="bar",
    figsize=(8,5)
)

plt.ylabel("Average Tip %")
plt.title("Average Tip Percentage by Passenger Count")
plt.show()

In [ ]:
hourly_tips = (
    tip_df.groupby("pickup_hour")
          ["tip_percentage"]
          .mean()
          .reset_index()
)

hourly_tips
hourly_tips.plot(
    x="pickup_hour",
    y="tip_percentage",
    kind="line",
    marker="o",
    figsize=(10,5)
)

plt.ylabel("Average Tip %")
plt.title("Average Tip Percentage by Hour")
plt.grid(True)

plt.show()


In [ ]:
print("Lowest tip by distance")
print(distance_tips.sort_values("tip_percentage"))

print("\nLowest tip by passenger count")
print(passenger_tips.sort_values("tip_percentage"))

print("\nLowest tip hours")
print(hourly_tips.sort_values("tip_percentage").head(10))

Additional analysis [optional]: Let's try comparing cases of low tips with cases of high tips to find out if we find a clear aspect that drives up the tipping behaviours

In [ ]:
# Compare trips with tip percentage < 10% to trips with tip percentage > 25%



<br>
Analyse the variation of passenger count across hours and days of the week.

In [ ]:
# See how passenger count varies across hours and days

hourly_passengers = (
    df.groupby("pickup_hour")["passenger_count"]
      .mean()
      .reset_index()
)

hourly_passengers.plot(
    x="pickup_hour",
    y="passenger_count",
    marker="o",
    figsize=(10, 5),
    title="Average Passenger Count by Hour"
)

plt.ylabel("Average Passenger Count")
plt.show()




<br>
Analyse the variation of passenger counts across zones

In [ ]:
# How does passenger count vary across zones



In [ ]:
# For a more detailed analysis, we can use the zones_with_trips GeoDataFrame
# Create a new column for the average passenger count in each zone.



Find out how often surcharges/extra charges are applied to understand their prevalance

<br>
Analyse the pickup/dropoff zones or times when extra charges are applied more frequently

In [ ]:
# How often is each surcharge applied?
[
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee"
]
df[
    [
        "extra",
        "mta_tax",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee"
    ]
].describe()

In [ ]:
surcharge_frequency = pd.DataFrame({
    "extra": (df["extra"] > 0).sum(),
    "mta_tax": (df["mta_tax"] > 0).sum(),
    "improvement_surcharge": (df["improvement_surcharge"] > 0).sum(),
    "congestion_surcharge": (df["congestion_surcharge"] > 0).sum(),
    "airport_fee": (df["airport_fee"] > 0).sum()
}, index=["count"]).T

surcharge_frequency.columns = ["trip_count"]

surcharge_frequency["percentage"] = (
    surcharge_frequency["trip_count"]
    / len(df)
    * 100
)

surcharge_frequency.sort_values(
    "percentage",
    ascending=False
)

In [ ]:
df["has_surcharge"] = (
    (df["extra"] > 0)
    | (df["mta_tax"] > 0)
    | (df["improvement_surcharge"] > 0)
    | (df["congestion_surcharge"] > 0)
    | (df["airport_fee"] > 0)
)

hourly_surcharge = (
    df.groupby("pickup_hour")["has_surcharge"]
      .mean()
      .mul(100)
      .reset_index()
)

hourly_surcharge.plot(
    x="pickup_hour",
    y="has_surcharge",
    marker="o",
    figsize=(10,5),
    title="Percentage of Trips With Surcharges by Hour"
)

plt.ylabel("% Trips With Surcharge")
plt.show()

In [ ]:
save_csv(df)

## Conclusion
<br>



### Final Insights and Recommendations
<br>

Conclude your analyses here. Include all the outcomes you found based on the analysis.

Based on the insights, frame a concluding story explaining suitable parameters such as location, time of the day, day of the week etc. to be kept in mind while devising a strategy to meet customer demand and optimise supply.

## Conclusion is:

### Demand Patterns
- Taxi demand varies significantly throughout the day.
- Pickup activity is lowest during early morning hours (around 4–5 AM).
- Demand increases sharply after 6 AM and remains high throughout the afternoon and evening.
- The busiest hour is around 6 PM, indicating strong evening commuting. Office finish hours.

### Revenue
- Daytime trips generate approx. 88% of total revenue and night trips contribute only about 12%.
- More number of trips lead to more amount of revenue.

### Trip characterstics
- Fare amount and trip duration are strogly correlated.
- Tip amount shows a medium positive correlation with trip distance.

### Passenger behaviours
- Most taxi rides are small group, 1 or 2 passengers.
- Passenger count doesn't impact fare amount.

### Geo analysis
- Airport zones are the most active pickup locations

### Vendor analysis
- Vendor 2 consistently charge higher fare then vendor 1 for short trips


<br>
Recommendations to optimize routing and dispatching based on demand patterns and operational inefficiencies

- Increase driver availability during morning and evening hours
- Schedule driver shifts that syncs with peak demand periods
- Use models to forecast peak demand position and position taxis there

<br>

Suggestions on strategically positioning cabs across different zones to make best use of insights uncovered by analysing trip trends across time, days and months.

<br>
Propose data-driven adjustments to the pricing strategy to maximize revenue while maintaining competitive rates with other vendors.

- Propose time based pricing for peak time.
- Adjust pricing in high demand zones.
- Vendor 1 can selectively increase rates.